In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip uninstall -y torchao
!pip install -q torchao==0.16.0

Found existing installation: torchao 0.16.0
Uninstalling torchao-0.16.0:
  Successfully uninstalled torchao-0.16.0


In [ ]:
import os

path = "/content/drive/MyDrive/BanglaVision/coco_20k_checkpoints/coco20k_step_1_20000"

print("Exists:", os.path.exists(path))
print("Files:", os.listdir(path))

Exists: True
Files: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'tokenizer.json', 'processor_config.json']


In [ ]:
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from peft import PeftModel
import torch, gc

MODEL_PATH = "/content/drive/MyDrive/BanglaVision/coco_20k_checkpoints/coco20k_step_1_20000"

torch.cuda.empty_cache()
gc.collect()

processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")

base_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    autocast_adapter_dtype=False
)

model.eval()

print("Model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

Model loaded


/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.language_model.model.decoder.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.language_model.model.decoder.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.language_model.model.decoder.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.language_model.model.decoder.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.language_model.model.decoder.layers.1.self_attn.v_proj.lora_A.default.weight', 'base_model.model.language_model.model.decoder.layers.1.self_attn.v_proj.lora_B.default.weight', 'base_model.model.language_model.model.decoder.layers.1.self_attn.q_proj.lora_A.default.weight', 'base_model.model.language_model.model.decoder.layers.1.self_attn.q_proj.lora_B.default.weight', 'base_model.model.language_model.model.decoder.layers.2.self_attn.v_proj.lora_A.default.w

In [ ]:
import torch
import re
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using Device:", device)

model = model.to(device)

Using Device: cuda


In [ ]:
def generate_caption(image):

    prompt = """
    Describe this image in detail.
    Mention people, objects, actions, and environment clearly.
    """

    inputs = processor(
        images=image,
        text=prompt,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=120,
            temperature=0.7
        )

    caption = processor.decode(output[0], skip_special_tokens=True)
    return caption

In [ ]:
def expand_caption(caption):

    prompt = f"""
    Expand this description with more details:

    {caption}
    """

    inputs = processor(text=prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=120
        )

    expanded = processor.decode(output[0], skip_special_tokens=True)
    return expanded

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

TRANSLATOR_MODEL = "facebook/nllb-200-distilled-600M"

translator_tokenizer = AutoTokenizer.from_pretrained(
    TRANSLATOR_MODEL
)

translator_model = AutoModelForSeq2SeqLM.from_pretrained(
    TRANSLATOR_MODEL,
    torch_dtype=torch.float16
).to(device)

print("Translator Loaded")

def en_to_bn(text):

    try:
        inputs = translator_tokenizer(
            text,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        translated_tokens = translator_model.generate(
            **inputs,
            forced_bos_token_id=translator_tokenizer.convert_tokens_to_ids("ben_Beng"),
            max_length=200
        )

        translated_text = translator_tokenizer.batch_decode(
            translated_tokens,
            skip_special_tokens=True
        )[0]

        return translated_text

    except Exception as e:

        print("Translation Error:", e)

        return text

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Translator Loaded


In [ ]:
def generate_caption(image):

    image = image.convert("RGB").resize((224, 224))

    prompt = """
    Describe this image clearly.
    Mention people, objects, actions, and environment.
    """

    inputs = processor(
        images=image,
        text=prompt,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=60,
            num_beams=3,
            do_sample=False
        )

    caption = processor.decode(
        output[0],
        skip_special_tokens=True
    ).strip()

    return caption

In [ ]:
def cultural_fix(text):

    replacements = {

        "bicycle": "রিকশা",
        "bike": "রিকশা",
        "man": "একজন মানুষ",
        "woman": "একজন নারী",
        "road": "রাস্তা",
        "street": "রাস্তা",

        "cloth": "লুঙ্গি",
        "towel": "গামছা"
    }

    for en, bn in replacements.items():

        text = text.replace(en, bn)

    return text

In [ ]:
def clean_bangla(text):

    text = text.strip()

    text = " ".join(text.split())

    if not text.endswith("।"):
        text += "।"

    return text

In [ ]:
import torch
import gc

def ask_ai(image, question_bn):

    if image is None:
        return "দয়া করে একটি ছবি আপলোড করুন।"

    try:
        image = image.convert("RGB")

        prompt = "Question: What is in this image? Answer:"

        inputs = processor(
            images=image,
            text=prompt,
            return_tensors="pt"
        )

        inputs = {
            k: v.to(device)
            for k, v in inputs.items()
        }

        with torch.no_grad():

            output = model.generate(
                **inputs,
                max_new_tokens=25,
                do_sample=False,
                num_beams=3
            )

        answer = processor.decode(
            output[0],
            skip_special_tokens=True
        )

        print("RAW OUTPUT:", answer)

        answer = answer.replace(prompt, "").strip()

        bad_outputs = [
            "",
            ".",
            "Answer:",
            "Question:",
            "unknown"
        ]

        if answer.lower() in bad_outputs:
            return "ছবিটি বোঝা যায়নি।"

        answer_bn = en_to_bn(answer)

        print("BANGLA:", answer_bn)

        answer_bn = clean_bangla(answer_bn)

        return answer_bn

    except Exception as e:

        print("ERROR:", e)

        return "অপ্রত্যাশিত সমস্যা হয়েছে।"

In [ ]:
import gradio as gr

gr.close_all()

custom_css = """
body {
    font-family: 'Inter', sans-serif;
    background: #0b1220;
    color: white;
}

.gradio-container {
    max-width: 900px !important;
    margin: auto;
}

h1 {
    text-align: center;
    color: #38bdf8;
}

.section {
    background: #111827;
    padding: 18px;
    border-radius: 12px;
    margin-bottom: 16px;
}

button {
    border-radius: 10px !important;
    font-size: 16px !important;
}
"""

def validated_ask(image, question):

    if image is None:
        return "দয়া করে একটি ছবি আপলোড করুন।"

    return ask_ai(image, question)

with gr.Blocks(css=custom_css) as demo:

    gr.Markdown("# BanglaVision AI")

    with gr.Group(elem_classes="section"):

        image_input = gr.Image(
            type="pil",
            label="Upload Image",
            height=300
        )

        question_input = gr.Textbox(
            label="Your Question (Bangla)",
            placeholder="উদাহরণ: এই ছবিতে কী আছে?",
            lines=3
        )

    with gr.Row():

        submit_btn = gr.Button("Generate Bengali Description")

        clear_btn = gr.Button("Clear")

    output = gr.Textbox(
        label="Output (Bangla)",
        lines=6
    )

    submit_btn.click(
        fn=validated_ask,
        inputs=[image_input, question_input],
        outputs=output
    )

    clear_btn.click(
        fn=lambda: (None, "", ""),
        inputs=[],
        outputs=[image_input, question_input, output]
    )

demo.launch(share=True)

/tmp/ipykernel_7428/789218324.py:42: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://18dda1a432b6c09a82.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
